# 17 · 4D multi-head attention: reshape Q, K and V / Atención multicabeza 4D: reorganiza Q, K y V

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/17-multi-head-attention.ipynb)

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#0ea5e9,rgba(14,165,233,0))"></div>

<span style="font:700 11px/1.6 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#0ea5e9">DEEP DIVE · TAKE-HOME · ABOUT 35 MINUTES / ESTUDIO A FONDO · PARA DESPUÉS · UNOS 35 MINUTOS</span>

Follow each token from (B, S, D) to (B, H, S, D_k), then compute attention without losing track of an axis. Synthetic examples, inline tests and pausable explorers run independently on CPU.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><div style="margin:0 0 0">Sigue cada token de (B, S, D) a (B, H, S, D_k) y calcula atención sin perder el significado de los ejes. Ejemplos sintéticos, pruebas integradas y exploradores con pausa funcionan de forma independiente en CPU.</div></div>

## What you will be able to do / Lo que podrás hacer

- Implement split_heads and test shape, values, contiguity and memory sharing.
- Compute attention scores, normalize over keys and concatenate head outputs.
- Verify NumPy results against PyTorch and inspect attention interactively.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><ul style="margin:0;padding-left:1.2em"><li style="margin:.35em 0">Implementar split_heads y comprobar forma, valores, contigüidad y memoria compartida.</li><li style="margin:.35em 0">Calcular puntuaciones de atención, normalizar sobre claves y concatenar cabezas.</li><li style="margin:.35em 0">Verificar NumPy con PyTorch e inspeccionar atención de forma interactiva.</li></ul></div>

<!-- CORE-PATH -->
**Core path · about 35 minutes.** Read the problem, run Setup, write the starter,
then check it before revealing the folded solution. Continue through NumPy
attention and the PyTorch cross-check. **Explore later:** widgets and GIF exports.

**Two outcomes:** track which token belongs to which head; distinguish a view's
axis order from its physical memory layout.

*Ruta esencial: predice los ejes, programa la función y comprueba sus valores.
Después compara NumPy con PyTorch; los exploradores son opcionales.*

## Problem: why give heads their own axis?
*Problema: separar cabezas permite calcular atención por cabeza y por lote.*

A Transformer first learns three projections of token embeddings:
`Q = X @ W_Q`, `K = X @ W_K`, and `V = X @ W_V`. Each has shape `(B, S, D)`.
A query asks what to retrieve, a key determines its match, and a value carries
what gets retrieved. We use small random projections for mechanics, not a trained
language model: these attention patterns have no learned semantic interpretation.

With `H` heads, split `D = H * D_k` features into groups. For example:

| Stage | Shape | Meaning |
|---|---|---|
| Projected Q, K or V | `(2, 6, 12)` | batch, token, features |
| Reshape | `(2, 6, 3, 4)` | batch, token, head, head features |
| Transpose | `(2, 3, 6, 4)` | batch, head, token, head features |
| Q @ K transpose | `(2, 3, 6, 6)` | batch, head, query token, key token |

Putting `(B, H)` first lets matrix multiplication batch over both axes. Each
head compares every query token with every key token in its own feature space.
The reshape does not learn the spaces: the projection matrices do.

$$
L_{bhst}=\frac{\sum_d Q_{bhsd}K_{bhtd}}{\sqrt{D_k}},\qquad
A_{bhst}=\operatorname{softmax}_{t}(L_{bhst}),\qquad
O_{bhsd}=\sum_t A_{bhst}V_{bhtd}.
$$

`L` contains scores (logits); `A` contains normalized weights. Division by
`sqrt(D_k)` counteracts the growth in dot-product variance when component
variances are roughly one. Softmax runs over **keys**, the last axis.
Heads enable distinct attention patterns; at fixed `D` they do not reduce the
leading `O(B * S² * D)` dot-product work, and scores occupy `B * H * S²` elements.

<div style="border-left:5px solid #0ea5e9;background:rgba(14,165,233,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#0ea5e9;margin-bottom:11px">TRACE IT FIRST · SÍGUELO PRIMERO</div>Will <code>x.reshape(B, H, S, D_k)</code> put the same token's features into each head? Trace <code>x[0, 1, 0]</code> <b>before you run anything</b>.</div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>¿<code>x.reshape(B, H, S, D_k)</code> mete las características del mismo token en cada cabeza? Sigue <code>x[0, 1, 0]</code> <b>antes de ejecutar nada</b>.</div>

## Setup
*Preparación: CPU, datos sintéticos y ninguna descarga de datos.*

Run in a standard Google Colab CPU runtime. NumPy, PyTorch, Matplotlib,
ipywidgets and Pillow are the only third-party packages. No GPU, data files,
API keys, FFmpeg or system installs are needed. In a minimal local Jupyter
runtime, install them with `%pip install -q numpy torch matplotlib ipywidgets pillow`.
The notebook uses plain assertions, so pytest is optional.

In [ ]:
from __future__ import annotations

from collections.abc import Callable
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
import ipywidgets as widgets
from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass  # Standard Jupyter already supports ipywidgets.

np.set_printoptions(precision=3, suppress=True)
plt.rcParams.update({"figure.dpi": 100, "font.size": 10})
print(f"NumPy {np.__version__} | PyTorch {torch.__version__} | CPU")

## Core activity: split heads
*Actividad esencial: conserva la identidad de cada token al separar las cabezas.*

**Predict → Run → Explain → Check / Predice → Ejecuta → Explica → Comprueba**

Edit and run the starter cell below. Then run the
inline test cell below **before** opening the solution. The test accepts your
function as an argument. For an instructor walkthrough, Run all executes the
folded reference solution and all tests; a clean notebook has no failing cells.

Input contract: a NumPy array with three non-empty axes; a positive integer
head count that divides `D`. Reject booleans as head counts. Preserve dtype
and values; do not mutate `x`. Non-contiguous inputs must also give correct
values; a no-copy guarantee is only demonstrated for the contiguous example.

**Hint:** the permutation is `(0, 2, 1, 3)`. Explain why axes 1 and 2 trade places.
After implementing your version, skip the folded cell and call
`test_split_heads(split_heads)` in the next code cell.

In [ ]:
def split_heads(x: np.ndarray, num_heads: int) -> np.ndarray:
    """Map (B, S, D) to (B, H, S, D // H) without mixing token features."""
    # TODO: validate three non-empty axes and a positive integer head count.
    # TODO: reject D % num_heads != 0.
    # TODO: reshape to (B, S, H, D_k), then transpose to (B, H, S, D_k).
    raise NotImplementedError("Complete reshape, then transpose")

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

def split_heads(x: np.ndarray, num_heads: int) -> np.ndarray:
    """Split features into heads and move the head axis before the token axis.

    For a C-contiguous input, reshape and transpose share the input's storage.
    The result need not be C-contiguous. Arbitrary strides are accepted; NumPy
    may copy when a requested reshape cannot be represented as a view.
    """
    if not isinstance(x, np.ndarray) or x.ndim != 3 or 0 in x.shape:
        raise ValueError("x must be a NumPy array with non-empty shape (B, S, D)")
    if isinstance(num_heads, (bool, np.bool_)) or not isinstance(num_heads, (int, np.integer)):
        raise TypeError("num_heads must be an integer, not a boolean")
    if num_heads <= 0 or x.shape[-1] % num_heads:
        raise ValueError("num_heads must be positive and divide D")
    batch, sequence, dim = x.shape
    return x.reshape(batch, sequence, num_heads, dim // num_heads).transpose(0, 2, 1, 3)

In [ ]:
def assert_raises(error: type[Exception], operation: Callable[[], object]) -> None:
    """Assert the specified exception without requiring pytest in the runtime."""
    try:
        operation()
    except error:
        return
    raise AssertionError(f"Expected {error.__name__}")


def test_split_heads(split: Callable[[np.ndarray, int], np.ndarray]) -> None:
    """Check explicit values, token identity, strides, aliases and bad inputs."""
    x = np.arange(24).reshape(2, 3, 4)
    original = x.copy()
    expected = np.array([
        [[[0, 1], [4, 5], [8, 9]], [[2, 3], [6, 7], [10, 11]]],
        [[[12, 13], [16, 17], [20, 21]], [[14, 15], [18, 19], [22, 23]]],
    ])
    actual = split(x, 2)
    assert actual.shape == (2, 2, 3, 2)
    assert actual.dtype == x.dtype
    np.testing.assert_array_equal(actual, expected)
    np.testing.assert_array_equal(x, original)
    # These layout assertions concern this non-degenerate, contiguous fixture.
    assert x.flags.c_contiguous
    assert not actual.flags.c_contiguous
    assert np.shares_memory(actual, x)
    assert actual.strides == tuple(s * x.itemsize for s in (12, 2, 4, 1))
    packed = np.ascontiguousarray(actual)
    assert packed.flags.c_contiguous and not np.shares_memory(packed, x)
    np.testing.assert_array_equal(packed, expected)
    # Correct shape alone does not catch direct-reshape mistakes.
    assert not np.array_equal(x.reshape(2, 2, 3, 2), expected)
    for candidate in (x, x[:, ::-1, :], x[:, :, ::-1], np.asfortranarray(x)):
        for heads in (1, 2, 4):
            result = split(candidate, heads)
            dk = candidate.shape[-1] // heads
            for b in range(2):
                for h in range(heads):
                    for s in range(3):
                        np.testing.assert_array_equal(result[b, h, s], candidate[b, s, h*dk:(h+1)*dk])
    singleton = split(np.ones((1, 1, 4)), 1)
    assert singleton.flags.c_contiguous  # A transpose is not ALWAYS non-contiguous.
    for heads in (0, -1, 3):
        assert_raises(ValueError, lambda: split(x, heads))
    for heads in (True, 2.0, "2"):
        assert_raises(TypeError, lambda: split(x, heads))
    for bad in (np.zeros((2, 4)), np.zeros((1, 0, 4))):
        assert_raises(ValueError, lambda: split(bad, 2))


test_split_heads(split_heads)
print("split_heads: shape, values, strides and input checks passed")

## Predict first / Predice primero

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#0ea5e9,rgba(14,165,233,0))"></div>

Someone says the following. **Decide whether they are right before you
reveal anything** — commit to one answer, then open the check.

<div style="border-left:5px solid #0ea5e9;background:rgba(14,165,233,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#0ea5e9;margin-bottom:11px">THE CLAIM · LA AFIRMACIÓN</div>&ldquo;The result has <b>exactly the right shape</b>, so the heads must be right too.&rdquo;</div>

A prediction you have committed to is worth more than one you keep
adjusting as the answer appears.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Alguien afirma que, como el resultado tiene <b>exactamente la forma correcta</b>, las cabezas también deben estarlo. Decide si tiene razón <b>antes</b> de revelar la comprobación.</div>


This claim is exactly what the attention stage's [heads picture](https://project-delphi.github.io/tensors-workshop/interactive/attention-stage.html?lang=en#heads) is built to test: switch its reshape control to the direct, untransposed one and the shape stays (2, 4, 4) while the traced cell comes from the wrong token.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Esta afirmación es justo lo que la <a href="https://project-delphi.github.io/tensors-workshop/interactive/attention-stage.html?lang=es#heads">imagen de cabezas</a> del escenario de atención está hecha para poner a prueba: cambia su control de reshape al directo, sin transpose, y la forma se mantiene en (2, 4, 4) mientras la celda marcada viene del token equivocado.</div>

In [ ]:
#@title 🤔 Predict: does the right shape mean the right heads? / Predice: ¿la forma correcta implica cabezas correctas? — run me / ejecútame { display-mode: 'form' }

# --- counterexample / contraejemplo
import numpy as np
example = np.arange(24).reshape(2, 3, 4)
correct = example.reshape(2, 3, 2, 2).transpose(0, 2, 1, 3)
wrong = example.reshape(2, 2, 3, 2)
assert wrong.shape == correct.shape
assert not np.array_equal(wrong, correct)
# --- end counterexample

# --- how the question is laid out / cómo se presenta la pregunta ---
# Radio buttons rather than a dropdown: one bilingual answer per line, with
# room around them, because the whole point is weighing the options against
# each other. Botones de opción en vez de un desplegable.
import contextlib
import html as pred_html
import io

PRED_ACCENT = "#0ea5e9"
PRED_SANS = "ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"
PRED_MONO = "ui-monospace,SFMono-Regular,Menlo,Consolas,monospace"


def pred_tag(text):
    """A small EN / ES marker, in words rather than in colour alone."""
    return (f'<span style="font:700 10px/1 {PRED_MONO};letter-spacing:.16em;'
            f'color:{PRED_ACCENT};opacity:.8;margin-right:9px;'
            f'vertical-align:.12em">{text}</span>')


def pred_is_measurement(line):
    """True for a printed reading, false for a sentence.

    A reading wants monospace and tight rows so the numbers line up under one
    another; a sentence wants prose type and room.
    """
    if ":" not in line:
        return False
    tail = line.rsplit(":", 1)[1].strip()
    return bool(tail) and (tail[0].isdigit()
                           or tail[0] in "[(-+."
                           or tail.startswith(("True", "False", "nan", "inf")))


def pred_panel(text):
    """The reveal, laid out instead of printed.

    Exactly the same words: `check_prediction` still prints, and this catches
    what it printed and gives it typography. EN and ES stay written out as
    tags rather than becoming a colour, because a reader who cannot see the
    colour still has to be able to tell the two apart.
    """
    blocks = []
    for line in text.rstrip("\n").split("\n"):
        stripped = line.strip()
        if not stripped:
            blocks.append('<div style="height:12px"></div>')
        elif stripped.startswith(("EN:", "ES:")):
            tag, body = stripped[:2], stripped[3:].strip()
            blocks.append(
                f'<p style="margin:.55em 0;font:400 15px/1.8 {PRED_SANS}">'
                f'{pred_tag(tag)}{pred_html.escape(body)}</p>')
        elif pred_is_measurement(stripped):
            blocks.append(
                f'<div style="font:400 13.5px/2.0 {PRED_MONO};'
                f'white-space:pre-wrap">{pred_html.escape(stripped)}</div>')
        else:
            blocks.append(
                f'<p style="margin:.55em 0;font:600 15.5px/1.75 {PRED_SANS}">'
                f'{pred_html.escape(stripped)}</p>')
    return (f'<div style="border-left:4px solid {PRED_ACCENT};'
            f'background:rgba(130,130,150,.08);border-radius:0 10px 10px 0;'
            f'padding:16px 20px;margin:.4em 0 0">{"".join(blocks)}</div>')


def pred_render(choice, reveal):
    """Run the check, catch what it prints, and show it laid out."""
    caught = io.StringIO()
    with contextlib.redirect_stdout(caught):
        check_prediction(choice, reveal)
    display(widgets.HTML(pred_panel(caught.getvalue())))


pred_choice = widgets.RadioButtons(
    options=[
        ("— choose one / elige una —", None),
        ("Yes — the shapes match, so the heads match / Sí — las formas coinciden, las cabezas también", 'same'),
        ("No — only reshape then transpose keeps a token together / No — solo reshape y luego transpose", 'transpose'),
        ("Neither array is valid / Ninguno de los dos es válido", 'neither'),
    ],
    value=None,
    description="",
    layout=widgets.Layout(width="auto", margin="0 0 6px 0"),
)

pred_reveal = widgets.Checkbox(
    value=False,
    description="Show me the answer / Muéstrame la respuesta",
    indent=False,
    layout=widgets.Layout(margin="10px 0 4px 0"),
)


def check_prediction(choice, reveal):
    if choice is None:
        print("Choose an answer first / Elige una respuesta primero.")
        return

    if not reveal:
        print("Answer saved / Respuesta guardada.")
        print("Tick the box above when you are ready / Marca la casilla de arriba cuando quieras.")
        return

    print("Shapes / Formas:", correct.shape, "==", wrong.shape)
    print()
    print("Head 0, token 1 should be / La cabeza 0, token 1 debe ser:", example[0, 1, :2])
    print("  reshape then transpose gives / reshape y transpose dan:", correct[0, 0, 1])
    print("  direct reshape gives / el reshape directo da:          ", wrong[0, 0, 1])
    print()
    if choice == "transpose":
        print("You were right / Acertaste.")
    else:
        print("You were wrong — read on / Te equivocaste; sigue leyendo.")
    print()
    print("EN: both arrays have the same shape; only reshape-then-transpose keeps one token's features inside one head. A direct reshape groups whatever sits next to it in memory.")
    print("ES: ambos arreglos tienen la misma forma; solo reshape seguido de transpose mantiene las características de un token dentro de una cabeza. Un reshape directo agrupa lo que esté contiguo en memoria.")


# The one <style> block in these notebooks, and the markdown rule does not
# cover it. ipywidgets gives no way to set the space between radio options
# from Python, and this is *widget output*, not a markdown cell: Colab strips
# <style> from markdown -- which is why every box in these notebooks is
# inline-styled -- but renders it in an output, the same path pandas' own
# Styler uses. Scoped to one added class so it can reach nothing else, and if
# it is ever dropped the options still work, just closer together.
pred_choice.add_class("pred-radio")

display(widgets.HTML(
    "<style>"
    ".pred-radio .widget-radio-box label{display:flex;align-items:flex-start;"
    "margin:0 0 13px;font:400 15px/1.6 " + PRED_SANS + "}"
    ".pred-radio input[type=radio]{flex:none;margin:4px 11px 0 0;"
    "transform:scale(1.15)}"
    "</style>"
))

pred_output = widgets.interactive_output(
    pred_render,
    {"choice": pred_choice, "reveal": pred_reveal},
)

pred_heading = widgets.HTML(
    f'<div style="font:700 11px/1.6 {PRED_MONO};letter-spacing:.18em;'
    f'color:{PRED_ACCENT};margin:2px 0 12px">'
    f'YOUR PREDICTION \u00b7 TU PREDICCI\u00d3N</div>'
)

display(widgets.VBox(
    [pred_heading, pred_choice, pred_reveal, pred_output],
    layout=widgets.Layout(padding="2px 0 14px 0"),
))


## Compute attention from Q, K and V
*Cálculo: normaliza sobre las claves y reúne las salidas de cada cabeza.*

The final two dimensions are the matrix axes. In `einsum`, `s` identifies
queries, `t` keys, and `d` head features; only `d` disappears in the score
contraction. We implement unmasked self-attention with no dropout. After weighting
V, transpose back **before** merging heads. A trained layer would then apply
an output projection `W_O`; this exercise stops at the concatenated heads.

<div style="border-left:5px solid #d97706;background:rgba(217,119,6,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#d97706;margin-bottom:11px">A TRAP · UNA TRAMPA</div>A transpose changes <b>strides</b>, not the storage order. Use <code>np.ascontiguousarray</code> when a consumer requires packed storage. In PyTorch call <code>.contiguous()</code> before an incompatible <code>.view()</code>; <code>.reshape()</code> copies instead, when it must.</div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Una transposición cambia los <b>strides</b>, no el orden en memoria. Usa <code>np.ascontiguousarray</code> cuando el consumidor necesite memoria compacta. En PyTorch llama a <code>.contiguous()</code> antes de un <code>.view()</code> incompatible; <code>.reshape()</code> copia cuando hace falta.</div>

In [ ]:
def scaled_attention(q: np.ndarray, k: np.ndarray, v: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return logits, key-normalized weights and values for 4D self-attention.

    Inputs are finite floating arrays with the same non-empty (B, H, S, D_k)
    shape. This teaching implementation has no masking or dropout.
    """
    if q.ndim != 4 or q.shape != k.shape or q.shape != v.shape or 0 in q.shape:
        raise ValueError("q, k and v must have the same non-empty (B, H, S, D_k) shape")
    if any(not np.issubdtype(a.dtype, np.floating) or not np.isfinite(a).all() for a in (q, k, v)):
        raise ValueError("q, k and v must contain finite floating-point values")
    logits = np.einsum("bhsd,bhtd->bhst", q, k) / np.sqrt(q.shape[-1])
    shifted = logits - logits.max(axis=-1, keepdims=True)
    weights = np.exp(shifted)
    weights /= weights.sum(axis=-1, keepdims=True)
    values = np.einsum("bhst,bhtd->bhsd", weights, v)
    return logits, weights, values


rng = np.random.default_rng(17)
B, S, D, H = 2, 6, 12, 3
X = rng.normal(size=(B, S, D))
W_Q, W_K, W_V = rng.normal(scale=1 / np.sqrt(D), size=(3, D, D))
Q, K, V = (X @ w for w in (W_Q, W_K, W_V))
qh, kh, vh = (split_heads(a, H) for a in (Q, K, V))
scores, weights, head_values = scaled_attention(qh, kh, vh)
merged = head_values.transpose(0, 2, 1, 3).reshape(B, S, D)
print("Q/K/V:", Q.shape, "→ heads:", qh.shape)
print("Scores:", scores.shape, "→ concatenated heads:", merged.shape)
print("dtype:", qh.dtype, "| head axis is 1, key axis is -1")


In [ ]:
def test_attention() -> None:
    """Verify scores by explicit loops, normalization, values and PyTorch."""
    target = np.empty((B, H, S, S))
    for b in range(B):
        for h in range(H):
            for s in range(S):
                for t in range(S):
                    target[b, h, s, t] = np.dot(qh[b, h, s], kh[b, h, t]) / np.sqrt(D // H)
    np.testing.assert_allclose(scores, target, atol=1e-12)
    np.testing.assert_allclose(scores, qh @ kh.swapaxes(-1, -2) / np.sqrt(D // H), atol=1e-12)
    np.testing.assert_allclose(weights.sum(axis=-1), 1, atol=1e-12)
    assert np.all(weights >= 0) and np.isfinite(weights).all()
    np.testing.assert_allclose(head_values, weights @ vh, atol=1e-12)
    np.testing.assert_allclose(merged, np.concatenate([head_values[:, h] for h in range(H)], axis=-1))
    # Zero logits must average values uniformly: an independent known answer.
    zero = np.zeros((1, 2, 3, 2))
    values = np.arange(12, dtype=float).reshape(zero.shape)
    _, uniform, averaged = scaled_attention(zero, zero, values)
    np.testing.assert_allclose(uniform, 1 / 3)
    np.testing.assert_allclose(averaged, np.broadcast_to(values.mean(axis=2, keepdims=True), values.shape))
    # Directly exercise the stable softmax with finite logits near 1e4.
    _, stable, _ = scaled_attention(zero + 100, zero + 100, values)
    assert np.isfinite(stable).all()
    np.testing.assert_allclose(stable, 1 / 3)

    qt, kt, vt = (torch.from_numpy(a).reshape(B, S, H, D // H).permute(0, 2, 1, 3) for a in (Q, K, V))
    assert not qt.is_contiguous()
    np.testing.assert_allclose(qt.numpy(), qh)
    result = F.scaled_dot_product_attention(qt, kt, vt, dropout_p=0.0)
    np.testing.assert_allclose(result.numpy(), head_values, rtol=1e-10, atol=1e-12)
    packed = result.transpose(1, 2).contiguous().view(B, S, D)
    assert packed.is_contiguous()
    np.testing.assert_allclose(packed.numpy(), merged, rtol=1e-10, atol=1e-12)


test_attention()
print("Attention: explicit-loop oracle, softmax and PyTorch checks passed")

## Core complete: explain the axis you changed
*Comprueba: forma correcta no implica valores correctos ni memoria contigua.*

For `(B, S, D) = (4, 10, 24)` and `H = 6`, write the head shape and score shape.
Why would softmax over `axis=-2` answer a different question? Why can direct
reshape pass a shape test but fail the value test?

<details><summary>Checkpoint answer / Respuesta</summary>

Heads: `(4, 6, 10, 4)`; scores: `(4, 6, 10, 10)`. The last score axis indexes
keys, so each query's weights must sum over that axis. A direct reshape groups
adjacent storage without exchanging the token and head axes.

</details>

## Explore later / Explora después

### Inspect a head and a query
*Explora una cabeza y una consulta; reproduce o pausa la secuencia.*

Move the head slider and step through query tokens. The highlighted row of the
heatmap supplies the coefficients in the value-weighted sum. The Play button
is pausable; the callback also works directly as `show_attention(0, 0)` if your
notebook viewer does not support widgets.

In [ ]:
#@title 🔍 Inspect a head and a query / Explora una cabeza y una consulta — run me / ejecútame { display-mode: 'form' }

def show_attention(head: int, query: int) -> None:
    """Inspect one query's attention and its weighted value contributions."""
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), constrained_layout=True)
    axes[0].imshow(weights[0, head], vmin=0, vmax=1, cmap="viridis")
    axes[0].axhline(query - 0.5, color="tomato", lw=2)
    axes[0].axhline(query + 0.5, color="tomato", lw=2)
    axes[0].set(title=f"Head {head}: all weights", xlabel="Key token", ylabel="Query token")
    axes[1].bar(np.arange(S), weights[0, head, query], color="#0f766e")
    axes[1].set(title=f"Query {query}: sums to 1", xlabel="Key token", ylim=(0, 1))
    contributions = weights[0, head, query, :, None] * vh[0, head]
    limit = max(float(np.abs(contributions).max()), 1e-9)
    axes[2].imshow(contributions, cmap="coolwarm", vmin=-limit, vmax=limit, aspect="auto")
    axes[2].set(title="Weighted V: sum rows → output", xlabel="Head feature", ylabel="Key token")
    plt.show()
    plt.close(fig)


head_slider = widgets.IntSlider(value=0, min=0, max=H-1, description="Head", continuous_update=False)
query_slider = widgets.IntSlider(value=0, min=0, max=S-1, description="Query", continuous_update=False)
query_play = widgets.Play(value=0, min=0, max=S-1, interval=800)
query_link = widgets.jslink((query_play, "value"), (query_slider, "value"))
attention_output = widgets.interactive_output(show_attention, {"head": head_slider, "query": query_slider})
display(widgets.VBox([head_slider, widgets.HBox([query_play, query_slider]), attention_output]))

### Three short animations, with local GIF export
*Tres animaciones: expórtalas localmente sin FFmpeg ni descargas.*

The previews below load from the workshop site when online. All figures and
widgets above work independently of those URLs. Run the optional export cell
to create the same GIFs yourself; Pillow writes them without a system encoder.
Use the Play/slider explorer above to pause and inspect a chosen state.
![Split animation for module 17](https://project-delphi.github.io/tensors-workshop/images/cube-17-split.gif)

![Scores animation for module 17](https://project-delphi.github.io/tensors-workshop/images/cube-17-scores.gif)

![Values animation for module 17](https://project-delphi.github.io/tensors-workshop/images/cube-17-values.gif)

In [ ]:
#@title 🎞️ Animation builder / Constructor de animaciones — run me / ejecútame { display-mode: 'form' }

def make_attention_animation(kind: str = "split") -> FuncAnimation:
    """Animate head assignment, score rows or weighted-value accumulation."""
    if kind not in ("split", "scores", "values"):
        raise ValueError("kind must be split, scores or values")
    x = np.arange(24).reshape(3, 8)
    heads = x.reshape(3, 2, 4).transpose(1, 0, 2)
    q = heads.astype(float) / 12
    logits = q @ q.swapaxes(-1, -2) / 2
    probs = np.exp(logits - logits.max(axis=-1, keepdims=True))
    probs /= probs.sum(axis=-1, keepdims=True)
    fig, axes = plt.subplots(1, 2, figsize=(8, 3.4), constrained_layout=True)

    def draw(frame: int) -> list:
        for ax in axes:
            ax.clear()
        head, query = (frame // 3) % 2, frame % 3
        if kind == "split":
            axes[0].imshow(x, cmap="viridis", vmin=0, vmax=23)
            axes[1].imshow(heads[head], cmap="viridis", vmin=0, vmax=23)
            for ax, data in ((axes[0], x), (axes[1], heads[head])):
                for row, col in np.ndindex(data.shape):
                    ax.text(col, row, str(data[row, col]), ha="center", va="center", color="white" if data[row, col] < 12 else "#18212f")
            axes[1].set_yticks(range(3))
            axes[1].set_xticks(range(4))
            axes[0].axvspan(head*4-0.5, head*4+3.5, color="orange", alpha=.25)
            axes[0].set(title="One batch: (S=3, D=8)", xlabel="Feature", ylabel="Token")
            axes[1].set(title=f"Head {head}: (S=3, D_k=4)", xlabel="Head feature", ylabel="Same token")
        elif kind == "scores":
            axes[0].imshow(probs[head], vmin=0, vmax=1, cmap="viridis")
            axes[0].axhline(query, color="orange", lw=2)
            axes[0].set(title=f"Head {head}: softmax(QKᵀ / √4)", xlabel="Key", ylabel="Query")
            axes[1].bar(range(3), probs[head, query], color="#0f766e")
            axes[1].set(title=f"Query {query}: weights sum to 1", xlabel="Key", ylim=(0, 1))
        else:
            stop = query + 1
            contributions = probs[head, 0, :, None] * q[head]
            axes[0].imshow(contributions, vmin=0, vmax=2, cmap="viridis", aspect="auto")
            axes[0].axhspan(-.5, stop-.5, color="orange", alpha=.25)
            axes[0].set(title=f"Head {head}, query 0: A × V", xlabel="Feature", ylabel="Key")
            axes[1].bar(range(4), contributions[:stop].sum(axis=0), color="#0f766e")
            axes[1].set(title=f"Sum first {stop}/3 keys", xlabel="Output feature", ylim=(0, 2))
        fig.suptitle("QKV: feature groups → token comparisons → weighted values")
        return []

    animation = FuncAnimation(fig, draw, frames=6, interval=900, blit=False)
    plt.close(fig)
    return animation

In [ ]:
EXPORT_GIFS = False  # Set True to write three GIF files to this runtime.
if EXPORT_GIFS:
    for kind in ('split', 'scores', 'values'):
        animation = make_attention_animation(kind)
        destination = Path(f"cube-17-{kind}.gif")
        animation.save(str(destination), writer=PillowWriter(fps=1.25), dpi=90)
        display(destination)

### API references
*Referencias de las bibliotecas.*

[NumPy transpose](https://numpy.org/doc/stable/reference/generated/numpy.transpose.html), [PyTorch scaled dot-product attention](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html).

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#0ea5e9,rgba(14,165,233,0))"></div>

## Done with this deep dive / Fin de este estudio a fondo

Next deep dive / Siguiente estudio a fondo: **18 · Latent feature compression with SVD / Compresión de características latentes con SVD** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/18-feature-compression.ipynb).

[← Workshop site / Sitio del taller](https://project-delphi.github.io/tensors-workshop/) · [All notebooks / Todos los notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook / Manual](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)